# 11 - Subject-Grouped Subtype Validation

This notebook replaces the preliminary image-stratified subtype estimates with subject-grouped evaluation. Subject identifiers were recovered from the refreshed NIST SD302 sources and matched to all accepted expert subtype labels.

The analysis keeps the frozen ResNet-18 embeddings and candidate classifiers from Notebook 09. The methodological change is that no subject may appear in both training and evaluation within a fold.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GROUPED_DIR = ROOT / "results" / "subtype_grouped_model_selection"
PREVIOUS_DIR = ROOT / "results" / "subtype_embedding_model_selection"

## Evaluation protocol

Five deterministic repetitions of five-fold `StratifiedGroupKFold` are used. Repetitions are accepted only when every test fold contains every target subtype. Feature scaling and classifier fitting occur inside each training fold.

In [ ]:
# Regenerate from the repository root when needed.
# !python scripts/select_subtype_grouped_models.py

folds = pd.read_csv(GROUPED_DIR / "grouped_fold_metrics.csv")
summary = pd.read_csv(GROUPED_DIR / "grouped_model_summary.csv")
per_class = pd.read_csv(GROUPED_DIR / "grouped_per_class_summary.csv")
cohorts = pd.read_csv(GROUPED_DIR / "grouped_cohort_summary.csv")

## Leakage and fold-completeness checks

These gates must pass before any grouped score is interpreted.

In [ ]:
assert folds["subject_overlap"].max() == 0
assert folds.groupby(["task", "model"]).size().eq(25).all()
assert cohorts.set_index("task").loc["arch", "subjects"] == 31
assert cohorts.set_index("task").loc["whorl", "subjects"] == 128

print("Grouped validation gates passed across", len(folds), "candidate-model folds.")

## Cohort

Accidental whorl remains excluded from model selection because only three accepted images from three subjects are available.

In [ ]:
cohorts[["task", "images", "subjects", "image_class_counts", "subject_class_counts"]]

## Candidate ranking

Macro F1 is the primary ranking metric because both tasks are imbalanced. Fold standard deviation describes instability across the repeated grouped partitions; it is not treated as an independent-sample confidence interval.

In [ ]:
ranking = summary.sort_values(["task", "macro_f1_mean"], ascending=[True, False])
ranking[["task", "model", "accuracy_mean", "balanced_accuracy_mean", "macro_f1_mean", "macro_f1_std"]]

## Statistical winner and deployable candidate

Linear SVC has the highest mean arch macro F1, but it does not provide probabilities. The existing application reports subtype probabilities, so balanced logistic regression at `C=1` remains the deployable arch candidate. Its grouped macro F1 is only 0.007 below the SVC result and well within fold variability. The whorl winner remains balanced logistic regression at `C=0.1`.

In [ ]:
statistical_winners = ranking.groupby("task", as_index=False).head(1)
deployment_models = {"arch": "logistic_C1", "whorl": "logistic_C0.1"}
deployment = pd.concat([
    summary[(summary["task"] == task) & (summary["model"] == model)]
    for task, model in deployment_models.items()
])
deployment[["task", "model", "balanced_accuracy_mean", "macro_f1_mean", "macro_f1_std"]]

## Effect of subject grouping

The grouped estimates are lower than the preliminary image-stratified values. This is evidence that repeated impressions from the same subjects made the preliminary task easier, and confirms why subject-level separation matters.

In [ ]:
previous = pd.read_csv(PREVIOUS_DIR / "embedding_model_selection_summary.csv")
comparison = deployment[["task", "model", "macro_f1_mean"]].rename(columns={"macro_f1_mean": "grouped_macro_f1"})
previous_selected = {"arch": ("logistic_C1", 0.618997), "whorl": ("logistic_C0.1", 0.464193)}
comparison["preliminary_macro_f1"] = comparison["task"].map(lambda task: previous_selected[task][1])
comparison["change_after_grouping"] = comparison["grouped_macro_f1"] - comparison["preliminary_macro_f1"]
comparison

## Per-class behavior

The majority classes remain much easier than tented arch and the two minority whorl subtypes. These class-level results justify retaining the expert-review warning in the application.

In [ ]:
selected_keys = set(zip(deployment["task"], deployment["model"]))
selected_class_results = per_class[
    per_class.apply(lambda row: (row["task"], row["model"]) in selected_keys, axis=1)
]
selected_class_results[["task", "label", "precision_mean", "recall_mean", "f1_score_mean", "f1_score_std"]]

## Save the deployment decision

This aggregate table records the models and grouped evidence that should appear in manifests, documentation, and the web application.

In [ ]:
decision_columns = ["task", "model", "accuracy_mean", "balanced_accuracy_mean", "macro_f1_mean", "macro_f1_std"]
deployment[decision_columns].to_csv(GROUPED_DIR / "deployment_candidate_summary.csv", index=False)
comparison.to_csv(GROUPED_DIR / "grouped_vs_preliminary.csv", index=False)

print("Saved grouped deployment decision tables.")

## Conclusion

Subject-grouped validation is now the governing subtype evidence. The deployed model families can remain logistic regression for probability-based reporting, but the application and manuscript must replace the preliminary macro F1 values with the grouped estimates and continue to describe minority subtype predictions as requiring expert review.